In [24]:
import gzip
import csv
import pandas as pd

In [25]:
matrix_path = "../data/raw/GSE103001_series_matrix.txt.gz"

In [39]:
metadata = {}

with gzip.open(matrix_path, "rt") as f:
    reader = csv.reader(f, delimiter="\t")

    for row in reader:
        if row and row[0].startswith("!Sample_"):
            metadata.setdefault(row[0], []).append(row[1:])

print("Metadata fields: ", len(metadata))
print("Samples: ", len(metadata["!Sample_geo_accession"][0]))

Metadata fields:  31
Samples:  44


In [27]:
for characteristic in metadata["!Sample_characteristics_ch1"]:
    print(
        characteristic[0],
        "|",
        characteristic[22]
    )

tissue nature: normal | tissue nature: primary invasive breast carcinoma
tumor receptors: / | tumor receptors: ER+/HER2-
bloom: / | bloom: 2
er subtype: / | er subtype: A
tnm: / | tnm: T2N1aM0


In [28]:
def classify_condition(tissue):
    tissue = tissue.lower()

    if tissue == "normal":
        return "normal"
    elif tissue == "primary invasive breast carcinoma":
        return "tumor"
    else:
        raise ValueError(f"Unexpected tissue type: {tissue}")

In [30]:
titles = metadata["!Sample_title"][0]
accessions = metadata["!Sample_geo_accession"][0]

# Create a DataFrame with accession and title columns
sample_df = pd.DataFrame({
    "accession": accessions,
    "title": titles
})

# Add sample characteristics to the DataFrame
for characteristic in metadata["!Sample_characteristics_ch1"]:
    key = characteristic[0].split(":")[0].strip()

    # Extract the value after the colon and strip whitespace
    values = [
        value.split(":", 1)[1].strip()
        for value in characteristic
    ]

    sample_df[key] = values

# Add a new column "condition" based on the "tissue nature" column
print(sample_df["tissue nature"].unique())
sample_df["condition"] = (
    sample_df["tissue nature"].apply(classify_condition)
)

# Extract patient ID from title, e.g. "Pat_12-02_tumor" -> "12-02"
sample_df["patient"] = (
    sample_df["title"]
    .str.replace(r"^Pat_", "", regex=True)
    .str.replace(r"_(normal|tumor)$", "", regex=True)
)

sample_df.head()

<StringArray>
['normal', 'primary invasive breast carcinoma']
Length: 2, dtype: str


,accession,title,tissue nature,tumor receptors,bloom,er subtype,tnm,condition,patient
0,GSM2752350,Pat_12-02_normal,normal,/,/,/,/,normal,12-02
1,GSM2752351,Pat_12-03_normal,normal,/,/,/,/,normal,12-03
2,GSM2752352,Pat_13-02_normal,normal,/,/,/,/,normal,13-02
3,GSM2752353,Pat_13-03_normal,normal,/,/,/,/,normal,13-03
4,GSM2752354,Pat_13-05_normal,normal,/,/,/,/,normal,13-05


In [31]:
tumors = sample_df[sample_df["condition"] == "tumor"]

print("Tumor samples: ", len(tumors))
print("\nReceptor status: ")
print(tumors["tumor receptors"].value_counts(dropna=False))

print("\nER subtype: ")
print(tumors["er subtype"].value_counts(dropna=False))

print("\nPatients with both conditions: ")
print(
    sample_df.groupby("patient")["condition"].nunique().value_counts()
)

Tumor samples:  22

Receptor status: 
tumor receptors
ER+/HER2-    22
Name: count, dtype: int64

ER subtype: 
er subtype
A    12
B    10
Name: count, dtype: int64

Patients with both conditions: 
condition
2    22
Name: count, dtype: int64


In [32]:
import tarfile

tarpath = "../data/raw/GSE103001_RAW.tar"

with tarfile.open(tarpath, "r") as tar:
    count_files = [
        member.name for member in tar.getmembers() if member.isfile() and member.name.endswith(".txt.gz")
    ]

print("Count files: ", len(count_files))
print("\nFirst 5 filenames: ")

for filename in count_files[:5]:
    print(filename)

Count files:  44

First 5 filenames: 
GSM2752350_BCTN12-02_STAR_htseqReverseGenes_EnsemblGRCh37-75.txt.gz
GSM2752351_BCTN12-03_STAR_htseqReverseGenes_EnsemblGRCh37-75.txt.gz
GSM2752352_BCTN13-02_STAR_htseqReverseGenes_EnsemblGRCh37-75.txt.gz
GSM2752353_BCTN13-03_STAR_htseqReverseGenes_EnsemblGRCh37-75.txt.gz
GSM2752354_BCTN13-05_STAR_htseqReverseGenes_EnsemblGRCh37-75.txt.gz



| Component | Meaning |
| -------- | -------- |
| GSM2752350  | GEO sample accession  | 
| BCTN12-02  | Sample identifier | 
| STAR  | RNA-seq alignment software | 
| htseqReverseGenes  | Gene-level counts generated using HTSeq with reverse-stranded counting | 
| EnsemblGRCh37-75  | Ensembl annotation release 75, based on GRCh37 | 


In [41]:
from pathlib import Path

# Extract GSM accessions from filenames
count_accessions = [
    Path(filename).name.split("_")[0] for filename in count_files
]

# Get GSM accessions from metadata
metadata_accessions = metadata["!Sample_geo_accession"][0]

# Compare
print("Count file accessions: ", len(count_accessions))
print("Metadata accessions: ", len(metadata_accessions))

print("\nDuplicate count accessions: ", len(count_accessions) - len(set(count_accessions)))

print("\nDuplicate metadata accessions: ", len(metadata_accessions) - len(set(metadata_accessions)))

missing_metadata = set(count_accessions) - set(metadata_accessions)
missing_counts =  set(metadata_accessions) - set(count_accessions)

print("\nMissing from metadata: ", len(missing_metadata), missing_metadata)
print("Missing from count files: ", len(missing_counts), missing_counts)

Count file accessions:  44
Metadata accessions:  44

Duplicate count accessions:  0

Duplicate metadata accessions:  0

Missing from metadata:  0 set()
Missing from count files:  0 set()


In [43]:
# Checking whether the descriptive sample identifiers match, 
# if not that could indicate a naming inconsistency, an incorrect mapping, 
# or something worth investigating before differential expression analysis.

sample_titles = metadata["!Sample_title"][0]

for accession, title in zip(metadata_accessions[:5], sample_titles[:5]):
    print(accession, ":", title)

GSM2752350 : Pat_12-02_normal
GSM2752351 : Pat_12-03_normal
GSM2752352 : Pat_13-02_normal
GSM2752353 : Pat_13-03_normal
GSM2752354 : Pat_13-05_normal


In [47]:
# Map GEO accessions to sample codes from count filenames
count_ids = {
    Path(filename).name.split("_")[0]:
    Path(filename).name.split("_")[1].removeprefix("BCTN").removeprefix("BCTP")
    for filename in count_files
}

# Map GEO acessions to sample codes from metadata titles
metadata_ids = {
    accession: title.removeprefix("Pat_").rsplit("_", 1)[0]
    for accession, title in zip(metadata_accessions, sample_titles)
}

# Compare sample codes for each GEO accession
mismatches = {
    accession: (count_ids[accession], metadata_ids[accession])
    for accession in count_ids
    if count_ids[accession] != metadata_ids[accession]
}

print("Sample identifier mismatches: ", len(mismatches))
print(mismatches)

Sample identifier mismatches:  0
{}


In [48]:
from collections import Counter

prefix_counts = Counter(
    Path(filename).name.split("_")[1][:4] for filename in count_files
)

print(prefix_counts)

Counter({'BCTN': 22, 'BCTP': 22})


In [51]:
# Extract condition from filename
filename_conditions = {
    Path(filename).name.split("_")[0]: (
        "normal" if Path(filename).name.split("_")[1].startswith("BCTN") else "tumor"
    )
    for filename in count_files
}

# Extract condition from metadata title
metadata_conditions = {
    accession: title.rsplit("_", 1)[-1].lower() for accession, title in zip(metadata_accessions, sample_titles)
}

# Compare conditions
condition_mismatches = {
    accession: (
        filename_conditions[accession],
        metadata_conditions[accession]
    )
    for accsession in filename_conditions
    if filename_conditions[accession] != metadata_conditions[accession]
}

print("Condition mismatches: ", len(condition_mismatches))
print(condition_mismatches)

Condition mismatches:  0
{}
